In [ ]:
%load_ext autoreload
%autoreload 2

! pip install /kaggle/input/litmodels-bolts/lightning_bolts-0.3.3-py3-none-any.whl

!conda install '/kaggle/input/pydicom-conda-helper/libjpeg-turbo-2.1.0-h7f98852_0.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/libgcc-ng-9.3.0-h2828fa1_19.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/gdcm-2.8.9-py37h500ead1_1.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/conda-4.10.1-py37h89c1867_0.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/certifi-2020.12.5-py37h89c1867_1.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/openssl-1.1.1k-h7f98852_0.tar.bz2' -c conda-forge -y

!pip install /kaggle/input/kerasapplications -q
!pip install /kaggle/input/efficientnet-keras-source-code/ -q --no-deps
!pip install '/kaggle/input/effdet-latestvinbigdata-wbf-fused/ensemble_boxes-1.0.4-py3-none-any.whl' --no-deps

## MMDetection compatible torch installation
!pip install '/kaggle/input/pytorch-170-cuda-toolkit-110221/torch-1.7.0+cu110-cp37-cp37m-linux_x86_64.whl' --no-deps
!pip install '/kaggle/input/pytorch-170-cuda-toolkit-110221/torchvision-0.8.1+cu110-cp37-cp37m-linux_x86_64.whl' --no-deps
!pip install '/kaggle/input/pytorch-170-cuda-toolkit-110221/torchaudio-0.7.0-cp37-cp37m-linux_x86_64.whl' --no-deps

## Compatible Cuda Toolkit installation
!mkdir -p /kaggle/tmp && cp /kaggle/input/pytorch-170-cuda-toolkit-110221/cudatoolkit-11.0.221-h6bb024c_0 /kaggle/tmp/cudatoolkit-11.0.221-h6bb024c_0.tar.bz2 && conda install /kaggle/tmp/cudatoolkit-11.0.221-h6bb024c_0.tar.bz2 -y --offline

## MMDetection Offline Installation
!pip install '/kaggle/input/mmdetectionv2140/addict-2.4.0-py3-none-any.whl' --no-deps
!pip install '/kaggle/input/mmdetectionv2140/yapf-0.31.0-py2.py3-none-any.whl' --no-deps
!pip install '/kaggle/input/mmdetectionv2140/terminal-0.4.0-py3-none-any.whl' --no-deps
!pip install '/kaggle/input/mmdetectionv2140/terminaltables-3.1.0-py3-none-any.whl' --no-deps
!pip install '/kaggle/input/mmdetectionv2140/mmcv_full-1_3_8-cu110-torch1_7_0/mmcv_full-1.3.8-cp37-cp37m-manylinux1_x86_64.whl' --no-deps
!pip install '/kaggle/input/mmdetectionv2140/pycocotools-2.0.2/pycocotools-2.0.2' --no-deps
!pip install '/kaggle/input/mmdetectionv2140/mmpycocotools-12.0.3/mmpycocotools-12.0.3' --no-deps


!cp -r /kaggle/input/mmdetectionv2140/mmdetection-2.14.0 /kaggle/working/
!mv /kaggle/working/mmdetection-2.14.0 /kaggle/working/mmdetection
%cd /kaggle/working/mmdetection
!pip install -e . --no-deps
%cd /kaggle/working/

# References

- https://www.kaggle.com/illidan7/siim-covid-19-classification-train
- https://www.kaggle.com/illidan7/litmodelscolab-evaluation/data
- https://www.kaggle.com/shivanandmn/efficientnet-pytorch-lightning-train-inference
- https://www.kaggle.com/illidan7/siim-covid-19-frankenstein/notebook
- https://www.kaggle.com/whitegg/inference-studyclass-baseline
- https://www.kaggle.com/sreevishnudamodaran/siim-effnetv2-l-cascadercnn-mmdetection-infer

# Load libraries

In [ ]:
import sys
sys.path.append('../input/timm-pytorch-image-models/pytorch-image-models-master')
sys.path.append('../usr/lib/siim_covid19_littrainers_py/')
sys.path.append('/kaggle/working/mmdetection')

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.metrics.functional import accuracy
from pytorch_lightning.callbacks import ModelCheckpoint

In [ ]:
import platform
import numpy as np 
import pandas as pd 
import os
from tqdm.notebook import tqdm
import cv2
import pydicom
import gdcm
import glob
import gc
from math import ceil
import matplotlib.pyplot as plt
from pydicom.pixel_data_handlers.util import apply_voi_lut
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import albumentations

# from sklearn.model_selection import StratifiedKFold, GroupKFold

import math
import psutil

DATA_PATH = '/kaggle/input/siim-covid19-detection/'


import timm

In [ ]:
from PIL import Image

import torch
torch.manual_seed(0)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# import timm

import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
# from pytorch_metric_learning import losses
from torch.optim import lr_scheduler

import warnings
warnings.simplefilter('ignore')

In [ ]:
sub_df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')

# Form study and image dataframes
sub_df['level'] = sub_df.id.map(lambda idx: idx[-5:])
study_df = sub_df[sub_df.level=='study'].rename({'id':'study_id'}, axis=1)
image_df = sub_df[sub_df.level=='image'].rename({'id':'image_id'}, axis=1)

dcm_path = glob.glob('/kaggle/input/siim-covid19-detection/test/**/*dcm', recursive=True)
test_meta = pd.DataFrame({'dcm_path':dcm_path})
test_meta['image_id'] = test_meta.dcm_path.map(lambda x: x.split('/')[-1].replace('.dcm', '')+'_image')
test_meta['study_id'] = test_meta.dcm_path.map(lambda x: x.split('/')[-3].replace('.dcm', '')+'_study')

study_df = study_df.merge(test_meta, on='study_id', how='left')
image_df = image_df.merge(test_meta, on='image_id', how='left')

# Remove duplicates study_ids from study_df
# study_df.drop_duplicates(subset="study_id",keep='first', inplace=True)

In [ ]:
# df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')
# if df.shape[0] == 2477:
#     fast_sub = True
#     fast_df = pd.DataFrame(([['00086460a852_study', 'negative 1 0 0 1 1'], 
#                          ['000c9c05fd14_study', 'negative 1 0 0 1 1'], 
#                          ['65761e66de9f_image', 'none 1 0 0 1 1'], 
#                          ['51759b5579bc_image', 'none 1 0 0 1 1']]), 
#                        columns=['id', 'PredictionString'])
#     study_df = study_df.sample(2)
#     image_df = image_df.sample(2)
# else:
#     fast_sub = False

# fast_sub = False

In [ ]:
df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')
if df.shape[0] == 2477:
    fast_sub = True
    df.to_csv("/kaggle/working/submission.csv", index=False)
else:
    fast_sub = False

# VICTOR'S MODELS

In [ ]:
if not fast_sub:
    !pip install /kaggle/input/boxescode/ensemble_boxes-1.0.6-py3-none-any.whl
    # Necessary for gdcm (pydicom)
    !pip install /kaggle/input/boxescode/Pillow-5.3.0-cp37-cp37m-manylinux1_x86_64.whl
    # (submission)
    !cp /kaggle/input/realboxescode/*.py . # Get the code
    
if not fast_sub:
#     from importlib import reload
    import shutil
    from pathlib import Path
    
if not fast_sub:
    import const
#     const = reload(const)
    import preprocess
#     preprocess = reload(preprocess)
    import make_data_class
#     make_data_class = reload(make_data_class)
    import make_data_detect
#     make_data_detect = reload(make_data_detect)

    const.DIR_ORIGINAL_DATA = "/kaggle/input/siim-covid19-detection/"
    const.DIR_WORKING = "/kaggle/working/"
    
if not fast_sub:
    preprocess.make_images("png", 512, test_only=True, cut_1263=False)
    
if not fast_sub:
    preprocess.make_id_orig_size_csv(test_only=True)
    
if not fast_sub:
    make_data_class.create_and_validate_data(
        src="png512", dst="png512", test_only=True
    )
    
if not fast_sub:
    const.DIR_BASE_MODELS = "../input/models-class"
    const.SUBDIR_BASE_MODELS_CLASS = ""
    const.SUBDIR_MODELS_CLASS = "../input/models-class"
    
if not fast_sub:
    # Prepare dirs for preds
    # Make pred dirs
    for d in ["preds_class"]:
        if not Path(d).exists():
            Path(d).mkdir()
            
if not fast_sub:
    # Necessary for *importing* fastai
    !pip install /kaggle/input/boxescode/Pillow-8.3.0-cp37-cp37m-manylinux_2_5_x86_64.manylinux1_x86_64.whl
    
if not fast_sub:
    !pip install /kaggle/input/boxescode/timm-0.4.12-py3-none-any.whl
    
if not fast_sub:
    import train_class
#     train_class = reload(train_class)

if not fast_sub:
    # Necessary for *using* fastai (specifically, dataloaders)
    !pip install /kaggle/input/boxescode/Pillow-5.3.0-cp37-cp37m-manylinux1_x86_64.whl
    
if not fast_sub:
    IMAGE_PATH = "png512"
    NUM_FOLDS = 5
    IMG_SIZE = 384
    MODEL_NAME = "swin_large_patch4_window12_384"
    const.DIR_BASE_MODELS = "../input/models-class"
    const.DIR_BASE_MODELS += "/base"  # Do /base for swin

    dls_folds = [
        train_class.get_dls(
            image_path=IMAGE_PATH, 
            img_size=IMG_SIZE, 
            is_neg=False, 
            test_only=True,
        )
        for fold in range(NUM_FOLDS)
    ]

    print("\n dls stats:")
    for dls in dls_folds:
        print(len(dls.train.items))
        print(len(dls.valid.items))
        print(dls.vocab) # Expect: atyp, ind, neg, typ

    learn_folds = [
        train_class.get_learn(dls=dls, model_name=MODEL_NAME, is_neg=False, load_model_fold=fold)
        for fold, dls in enumerate(dls_folds)
    ]

    with torch.no_grad():
        preds_test1, _targs_test = train_class.predict_and_save(
            learn=learn_folds,
            sname="test",
            model_name=MODEL_NAME,
            n_tta=1 if df.shape[0] == 2477 else 6,
            is_neg=False,
        )
    print("\n preds stats:")
    print(preds_test1.shape)

    preds_swin = preds_test1
    
    for learn in learn_folds:
        del learn.model
    torch.cuda.empty_cache()

In [ ]:
if not fast_sub:
    IMAGE_PATH = "png512"
    NUM_FOLDS = 5
    IMG_SIZE = 512
    MODEL_NAME = "tf_efficientnet_b3_ns"
    const.DIR_BASE_MODELS = "../input/models-class"
    const.DIR_BASE_MODELS += f"/{MODEL_NAME}"  # Do /base for swin

    dls_folds = [
        train_class.get_dls(
            image_path=IMAGE_PATH, 
            img_size=IMG_SIZE, 
            is_neg=False, 
            test_only=True,
        )
        for fold in range(NUM_FOLDS)
    ]

    print("\n dls stats:")
    for dls in dls_folds:
        print(len(dls.train.items))
        print(len(dls.valid.items))
        print(dls.vocab) # Expect: atyp, ind, neg, typ

    learn_folds = [
        train_class.get_learn(dls=dls, model_name=MODEL_NAME, is_neg=False, load_model_fold=fold)
        for fold, dls in enumerate(dls_folds)
    ]

    with torch.no_grad():
        preds_test1, _targs_test = train_class.predict_and_save(
            learn=learn_folds,
            sname="test",
            model_name=MODEL_NAME,
            n_tta=1 if df.shape[0] == 2477 else 6,
            is_neg=False,
        )
    print("\n preds stats:")
    print(preds_test1.shape)

    preds_eff3 = preds_test1
    
    for learn in learn_folds:
        del learn.model
    torch.cuda.empty_cache()

In [ ]:
if not fast_sub:
    im2study = preprocess.make_img_to_study_map(test_only=True)

In [ ]:
if not fast_sub:
    swin_preds = pd.read_csv("preds_class/swin_large_patch4_window12_384/test.csv")
    swin_preds = swin_preds.rename(columns={"Unnamed: 0": "study_id"})
    swin_preds.study_id = swin_preds.study_id.map(im2study)
    swin_preds = swin_preds.groupby("study_id").mean()

    eff3_preds = pd.read_csv("preds_class/tf_efficientnet_b3_ns/test.csv")
    eff3_preds = eff3_preds.rename(columns={"Unnamed: 0": "study_id"})
    eff3_preds.study_id = eff3_preds.study_id.map(im2study)
    eff3_preds = eff3_preds.groupby("study_id").mean()

    """
    NOTE TO SHYAM: swin_preds and eff3_preds look like this:
    (note that the index is study_id)

                      atyp       ind       neg       typ
    study_id                                            
    0107f2d291d6  0.022844  0.092874  0.821179  0.063103
    0847751da0f7  0.010139  0.058066  0.001401  0.930395
    19c66935e737  0.051818  0.169776  0.009698  0.768708
    1a3ded50a43c  0.001446  0.020516  0.000570  0.977467
    1d2ad8808b78  0.083540  0.216391  0.024317  0.675752
    ...                ...       ...       ...       ...
    ed55cb404718  0.236947  0.211894  0.418943  0.132216
    ee7780677a6a  0.014749  0.144133  0.001292  0.839826
    f146eb3c25c2  0.037931  0.157704  0.013192  0.791173
    f85b5d51e41d  0.005765  0.043988  0.000325  0.949922
    f8dbe97a5542  0.009323  0.049232  0.926429  0.015016
    """

# Config

In [ ]:
class Config:
    train_pcent = 0.85
    model_name = 'resnet50'
    image_size = (400, 400)
    num_classes = 4
    batch_size = 32
    num_workers = 8
    NB_EPOCHS = 30
    scaler = GradScaler()
    scheduler = 'CosineAnnealingLR'
    weight_decay = 1e-6
    T_max = 10
    min_lr = 1e-6
    lr = 1e-5

# Dataset - setup

In [ ]:
if not fast_sub:
    import numpy as np
    import pydicom
    from pydicom.pixel_data_handlers.util import apply_voi_lut

    def read_xray(path, voi_lut = True, fix_monochrome = True):
        # Original from: https://www.kaggle.com/raddar/convert-dicom-to-np-array-the-correct-way
        dicom = pydicom.read_file(path)

        # VOI LUT (if available by DICOM device) is used to transform raw DICOM data to 
        # "human-friendly" view
        if voi_lut:
            data = apply_voi_lut(dicom.pixel_array, dicom)
        else:
            data = dicom.pixel_array

        # depending on this value, X-ray may look inverted - fix that:
        if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
            data = np.amax(data) - data

        data = data - np.min(data)
        data = data / np.max(data)
        data = (data * 255).astype(np.uint8)

        return data

In [ ]:
if not fast_sub:
    def resize(array, size, keep_ratio=False, resample=Image.LANCZOS):
        # Original from: https://www.kaggle.com/xhlulu/vinbigdata-process-and-resize-to-image
        im = Image.fromarray(array)

        if keep_ratio:
            im.thumbnail((size, size), resample)
        else:
            im = im.resize((size, size), resample)

        return im

# Dataset: Effnb7

In [ ]:
if not fast_sub:
    split = 'test'
    save_dir = f'/kaggle/tmp/{split}/'

    os.makedirs(save_dir, exist_ok=True)

    save_dir = f'/kaggle/tmp/{split}/study/'
    os.makedirs(save_dir, exist_ok=True)

    if fast_sub:

        xray = read_xray('/kaggle/input/siim-covid19-detection/train/00086460a852/9e8302230c91/65761e66de9f.dcm')
        im = resize(xray, size=600)  
        study = '00086460a852' + '_study.png'
        im.save(os.path.join(save_dir, study))
        xray = read_xray('/kaggle/input/siim-covid19-detection/train/000c9c05fd14/e555410bd2cd/51759b5579bc.dcm')
        im = resize(xray, size=600)  
        study = '000c9c05fd14' + '_study.png'
        im.save(os.path.join(save_dir, study))
    else:

        for dirname, _, filenames in tqdm(os.walk(f'/kaggle/input/siim-covid19-detection/{split}')):
            for file in filenames:
                # set keep_ratio=True to have original aspect ratio
                xray = read_xray(os.path.join(dirname, file))
                im = resize(xray, size=600)  
                study = dirname.split('/')[-2] + '_study.png'
                im.save(os.path.join(save_dir, study))

In [ ]:
if not fast_sub:
    split = 'test'

    image_id = []
    dim0 = []
    dim1 = []
    splits = []
    save_dir = f'/kaggle/tmp/{split}/image/'
    os.makedirs(save_dir, exist_ok=True)

    if fast_sub:
        xray = read_xray('/kaggle/input/siim-covid19-detection/train/00086460a852/9e8302230c91/65761e66de9f.dcm')
        im = resize(xray, size=512)  
        im.save(os.path.join(save_dir,'65761e66de9f_image.png'))
        image_id.append('65761e66de9f.dcm'.replace('.dcm', ''))
        dim0.append(xray.shape[0])
        dim1.append(xray.shape[1])
        splits.append(split)
        xray = read_xray('/kaggle/input/siim-covid19-detection/train/000c9c05fd14/e555410bd2cd/51759b5579bc.dcm')
        im = resize(xray, size=512)  
        im.save(os.path.join(save_dir, '51759b5579bc_image.png'))
        image_id.append('51759b5579bc.dcm'.replace('.dcm', ''))
        dim0.append(xray.shape[0])
        dim1.append(xray.shape[1])
        splits.append(split)
    else:
        for dirname, _, filenames in tqdm(os.walk(f'/kaggle/input/siim-covid19-detection/{split}')):
            for file in filenames:
                # set keep_ratio=True to have original aspect ratio
                xray = read_xray(os.path.join(dirname, file))
                im = resize(xray, size=512)  
                im.save(os.path.join(save_dir, file.replace('.dcm', '_image.png')))
                image_id.append(file.replace('.dcm', ''))
                dim0.append(xray.shape[0])
                dim1.append(xray.shape[1])
                splits.append(split)
    meta = pd.DataFrame.from_dict({'image_id': image_id, 'dim0': dim0, 'dim1': dim1, 'split': splits})

# Dataset - EffNetV2_L

In [ ]:
if not fast_sub:
    STUDY_DIMS = (768, 768)
    IMAGE_DIMS = (512, 512)

    study_dir = f'/kaggle/tmp/test1/study/'
    os.makedirs(study_dir, exist_ok=True)

#     image_dir = f'/kaggle/tmp/test1/image/'
#     os.makedirs(image_dir, exist_ok=True)

    for index, row in tqdm(study_df[['image_id', 'dcm_path']].iterrows(), total=study_df.shape[0]):
        # set keep_ratio=True to have original aspect ratio
        xray = read_xray(row['dcm_path'])
        im = resize(xray, size=STUDY_DIMS[0])
        im.save(os.path.join(study_dir, row['image_id']+'.png'))

#     image_df['dim0'] = -1
#     image_df['dim1'] = -1

#     for index, row in tqdm(image_df[['image_id', 'dcm_path', 'dim0', 'dim1']].iterrows(), total=image_df.shape[0]):
#         # set keep_ratio=True to have original aspect ratio
#         xray = read_xray(row['dcm_path'])
#         im = resize(xray, size=IMAGE_DIMS[0])  
#         im.save(os.path.join(image_dir, row['image_id']+'.png'))
#         image_df.loc[image_df.image_id==row.image_id, 'dim0'] = xray.shape[0]
#         image_df.loc[image_df.image_id==row.image_id, 'dim1'] = xray.shape[1]

In [ ]:
if not fast_sub:
    study_df['image_path'] = study_dir+study_df['image_id']+'.png'
#     image_df['image_path'] = image_dir+image_df['image_id']+'.png'

In [ ]:
if not fast_sub:
    import os

    import efficientnet.tfkeras as efn
    import numpy as np
    import pandas as pd
    import tensorflow as tf

    def auto_select_accelerator():
        try:
            tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
            tf.config.experimental_connect_to_cluster(tpu)
            tf.tpu.experimental.initialize_tpu_system(tpu)
            strategy = tf.distribute.experimental.TPUStrategy(tpu)
            print("Running on TPU:", tpu.master())
        except ValueError:
            strategy = tf.distribute.get_strategy()
        print(f"Running on {strategy.num_replicas_in_sync} replicas")

        return strategy


    def build_decoder(with_labels=True, target_size=STUDY_DIMS, ext='png'):
        def decode(path):
            file_bytes = tf.io.read_file(path)
            if ext == 'png':
                img = tf.image.decode_png(file_bytes, channels=3)
            elif ext in ['jpg', 'jpeg']:
                img = tf.image.decode_jpeg(file_bytes, channels=3)
            else:
                raise ValueError("Image extension not supported")

            img = tf.cast(img, tf.float32) / 255.0
            img = tf.image.resize(img, target_size)

            return img

        def decode_with_labels(path, label):
            return decode(path), label

        return decode_with_labels if with_labels else decode

    def build_augmenter(with_labels=False):
        def augment(img):
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_saturation(img, SATURATION[0], SATURATION[1])
            img = tf.image.random_contrast(img, CONTRAST[0], CONTRAST[1])
            img = tf.image.random_brightness(img, BRIGHTNESS)
            return img

        def augment_with_labels(img, label):
            return augment(img), label

        return augment_with_labels if with_labels else augment

    def build_dataset(paths, labels=None, bsize=32, cache=True,
                      decode_fn=None, augment_fn=None,
                      augment=True, repeat=True, shuffle=1024, 
                      cache_dir=""):
        if cache_dir != "" and cache is True:
            os.makedirs(cache_dir, exist_ok=True)

        if decode_fn is None:
            decode_fn = build_decoder(labels is not None)

        if augment_fn is None:
            augment_fn = build_augmenter(labels is not None)

        AUTO = tf.data.experimental.AUTOTUNE
        slices = paths if labels is None else (paths, labels)

        dset = tf.data.Dataset.from_tensor_slices(slices)
        dset = dset.map(decode_fn, num_parallel_calls=AUTO)
        dset = dset.cache(cache_dir) if cache else dset
        dset = dset.map(augment_fn, num_parallel_calls=AUTO) if augment else dset
        dset = dset.map(transform, num_parallel_calls=AUTO) if augment else dset
        dset = dset.repeat() if repeat else dset
        dset = dset.shuffle(shuffle) if shuffle else dset
        dset = dset.batch(bsize).prefetch(AUTO)

        return dset
    #COMPETITION_NAME = "siim-cov19-test-img512-study-600"

In [ ]:
### START HERE ###

# Inference - EffNetV2_L

In [ ]:
import math
import tensorflow.keras.backend as K

SATURATION  = (0.9, 1.1)
CONTRAST = (0.9, 1.1)
BRIGHTNESS  =  0.1
ROTATION    = 10.0
SHEAR    = 2.0
HZOOM  = 8.0
WZOOM  = 4.0
HSHIFT = 4.0
WSHIFT = 4.0

def get_mat(rotation, shear, height_zoom, width_zoom, height_shift, width_shift):
    # returns 3x3 transformmatrix which transforms indicies
        
    # CONVERT DEGREES TO RADIANS
    rotation = math.pi * rotation / 180.
    shear    = math.pi * shear    / 180.

    def get_3x3_mat(lst):
        return tf.reshape(tf.concat([lst],axis=0), [3,3])
    
    # ROTATION MATRIX
    c1   = tf.math.cos(rotation)
    s1   = tf.math.sin(rotation)
    one  = tf.constant([1],dtype='float32')
    zero = tf.constant([0],dtype='float32')
    
    rotation_matrix = get_3x3_mat([c1,   s1,   zero, 
                                   -s1,  c1,   zero, 
                                   zero, zero, one])    
    # SHEAR MATRIX
    c2 = tf.math.cos(shear)
    s2 = tf.math.sin(shear)    
    
    shear_matrix = get_3x3_mat([one,  s2,   zero, 
                                zero, c2,   zero, 
                                zero, zero, one])        
    # ZOOM MATRIX
    zoom_matrix = get_3x3_mat([one/height_zoom, zero,           zero, 
                               zero,            one/width_zoom, zero, 
                               zero,            zero,           one])    
    # SHIFT MATRIX
    shift_matrix = get_3x3_mat([one,  zero, height_shift, 
                                zero, one,  width_shift, 
                                zero, zero, one])
    
    return K.dot(K.dot(rotation_matrix, shear_matrix), 
                 K.dot(zoom_matrix,     shift_matrix))


def transform(image):
    # input image - is one image of size [dim,dim,3] not a batch of [b,dim,dim,3]
    # output - image randomly rotated, sheared, zoomed, and shifted
    DIM = STUDY_DIMS[0]
    XDIM = DIM%2
    
    rot = ROTATION * tf.random.normal([1], dtype='float32')
    shr = SHEAR * tf.random.normal([1], dtype='float32')
    h_zoom = 1.0 + tf.random.normal([1], dtype='float32') / HZOOM
    w_zoom = 1.0 + tf.random.normal([1], dtype='float32') / WZOOM
    h_shift = HSHIFT * tf.random.normal([1], dtype='float32')
    w_shift = WSHIFT * tf.random.normal([1], dtype='float32')
  
    # GET TRANSFORMATION MATRIX
    m = get_mat(rot, shr, h_zoom, w_zoom, h_shift, w_shift) 

    # LIST DESTINATION PIXEL INDICES
    x = tf.repeat( tf.range(DIM//2,-DIM//2,-1), DIM )
    y = tf.tile( tf.range(-DIM//2,DIM//2),[DIM] )
    z = tf.ones([DIM*DIM],dtype='int32')
    idx = tf.stack( [x,y,z] )
    
    # ROTATE DESTINATION PIXELS ONTO ORIGIN PIXELS
    idx2 = K.dot(m,tf.cast(idx,dtype='float32'))
    idx2 = K.cast(idx2,dtype='int32')
    idx2 = K.clip(idx2,-DIM//2+XDIM+1,DIM//2)
    
    # FIND ORIGIN PIXEL VALUES           
    idx3 = tf.stack( [DIM//2-idx2[0,], DIM//2-1+idx2[1,]] )
    d = tf.gather_nd(image,tf.transpose(idx3))
        
    return tf.reshape(d,[DIM,DIM,3])

In [ ]:
if not fast_sub:
    import tensorflow as tf
    import tensorflow_hub as tfhub

    MODEL_ARCH = 'efficientnetv2-l-21k-ft1k'
    # Get the TensorFlow Hub model URL
    hub_type = 'feature_vector' # ['classification', 'feature_vector']
    MODEL_ARCH_PATH = f'/kaggle/input/efficientnetv2-tfhub-weight-files/tfhub_models/{MODEL_ARCH}/{hub_type}'

    # Custom wrapper class to load the right pretrained weights explicitly from the local directory
    class KerasLayerWrapper(tfhub.KerasLayer):
        def __init__(self, handle, **kwargs):
            handle = tfhub.KerasLayer(tfhub.load(MODEL_ARCH_PATH))
            super().__init__(handle, **kwargs)

In [ ]:
if not fast_sub:
    MODEL_PATH = '/kaggle/input/siim-effnetv2-keras-study-train-tpu-cv0-805'
    test_paths = study_df.image_path.tolist()
    BATCH_SIZE = 16

In [ ]:
if not fast_sub:
   # num_augs sets the number of image augmentations that are used
    # - fewer augs is faster
    # - more augs gives higher mAP
    #   - I have included the mAPs below (calculated using my custom script)
    num_augs_to_main_dups = {
        0: 0,  # 0 augs => mAP 0.3796
        1: 3,  # 1 augs => mAP 0.3806 (big improvement!)
        2: 7,  # 2 augs => mAP 0.3807
        4: 6,  # 4 augs => mAP 0.3814 (big improvement!)
        6: 7,  # 6 augs => mAP 0.3815
    }

    NUM_AUGS = 4
    MAIN_DUPS = num_augs_to_main_dups[NUM_AUGS]

    label_cols = ['negative', 'typical', 'indeterminate', 'atypical']
    study_df[label_cols] = 0

    test_decoder = build_decoder(with_labels=False,
                                 target_size=(STUDY_DIMS[0],
                                              STUDY_DIMS[0]), ext='png')
    test_dataset = build_dataset(
        test_paths, bsize=BATCH_SIZE, repeat=False, 
        shuffle=False, augment=False, cache=False,
        decode_fn=test_decoder
    )
    aug_datasets = [
        build_dataset(
            test_paths, bsize=BATCH_SIZE, repeat=False, 
            shuffle=False, augment=True, cache=False,
            decode_fn=test_decoder
        ) for _ in range(NUM_AUGS)
    ]
    all_datasets = [test_dataset] + aug_datasets

In [ ]:
if not fast_sub:
    with tf.device('/device:GPU:0'):
        models = []
        models0 = tf.keras.models.load_model(f'{MODEL_PATH}/model0.h5',
                                             custom_objects={'KerasLayer': KerasLayerWrapper})
        models1 = tf.keras.models.load_model(f'{MODEL_PATH}/model1.h5',
                                             custom_objects={'KerasLayer': KerasLayerWrapper})
        models2 = tf.keras.models.load_model(f'{MODEL_PATH}/model2.h5',
                                             custom_objects={'KerasLayer': KerasLayerWrapper})
        models3 = tf.keras.models.load_model(f'{MODEL_PATH}/model3.h5',
                                             custom_objects={'KerasLayer': KerasLayerWrapper})
        models4 = tf.keras.models.load_model(f'{MODEL_PATH}/model4.h5',
                                             custom_objects={'KerasLayer': KerasLayerWrapper})
        models.append(models0)
        models.append(models1)
        models.append(models2)
        models.append(models3)
        models.append(models4)
        
    if df.shape[0] == 2477:
        models = models[:1]

In [ ]:
# models = models[0]

if not fast_sub:
    study_df_ls = []
    for i, dset in enumerate(all_datasets):
        study_df_i = study_df.copy()
        study_df_i[label_cols] = sum([model.predict(dset, verbose=1) for model in models]) / len(models)
        study_df_ls.append(study_df_i)
        if i == 0:
            for _ in range(MAIN_DUPS):
                study_df_ls.append(study_df_i.copy())
    study_df = pd.concat(study_df_ls, 0)
    print(study_df.shape, 6334 * (1 + MAIN_DUPS + NUM_AUGS))
    
    grouped = study_df.groupby('study_id')[label_cols].mean()
    effv2l_preds = grouped # This includes all folds, averaged
    
study_df.drop_duplicates(subset="study_id",keep='first', inplace=True)

# if not fast_sub:
#     pred_str = []
#     for study_id in study_df.study_id:
#         row = grouped.loc[study_id]
#         pred_str.append(f'negative {row.negative} 0 0 1 1 typical {row.typical} 0 0 1 1 indeterminate {row.indeterminate} 0 0 1 1 atypical {row.atypical} 0 0 1 1')
#     study_df['PredictionString'] = pred_str

"""
NOTE TO SHYAM:

The variable effv2l_preds will look like this. It has one row for
each study. "study_id" is the index column

                    negative   typical  indeterminate  atypical
study_id                                                       
00188a671292_study  0.913999  0.016243       0.044692  0.025066
004bd59708be_study  0.000115  0.910569       0.075483  0.013833
00508faccd39_study  0.857553  0.039355       0.080776  0.022316
006486aa80b2_study  0.271177  0.229648       0.356773  0.142402
00655178fdfc_study  0.518083  0.160707       0.252867  0.068343
...                      ...       ...            ...       ...
ff1ba0e9aaf0_study  0.278385  0.321775       0.318428  0.081412
ff2cc4de58c5_study  0.433201  0.150897       0.288629  0.127273
ff2f0a744930_study  0.000155  0.968068       0.028076  0.003701
ff88940dce8b_study  0.514029  0.245264       0.206712  0.033995
fff7ef24961f_study  0.269686  0.486479       0.206711  0.037123
"""    

In [ ]:
if not fast_sub:
    del models
    del models0, models1, models2, models3, models4
    del test_dataset, aug_datasets, all_datasets, test_decoder
    _ = gc.collect()
    torch.cuda.empty_cache()

# Inference data loader effnb7

In [ ]:
if not fast_sub:
    class SIIMCOVID19TestData(Dataset):
        def __init__(self):
            super().__init__()

            if fast_sub:
                self.submission_df = fast_df.copy()
            else:
                self.submission_df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')

            self.submission_df['StudyInstanceUID'] = self.submission_df['id'].apply(lambda x: x.replace('_study', ''))
            self.submission_df = self.submission_df[~self.submission_df['StudyInstanceUID'].str.endswith('_image')]

            if fast_sub:
                paths = ['/kaggle/input/siim-covid19-detection/train/00086460a852/9e8302230c91/65761e66de9f.dcm',
                         '/kaggle/input/siim-covid19-detection/train/000c9c05fd14/e555410bd2cd/51759b5579bc.dcm']

            else:

                TEST_DIR = "/kaggle/input/siim-covid19-detection/test/"

                # Make a path folder
                paths = []
                for instance_id in tqdm(self.submission_df['StudyInstanceUID']):
                    paths.append(glob.glob(os.path.join(TEST_DIR, instance_id +"/*/*"))[0])


            self.submission_df['path'] = paths

            self.transform = transforms.Compose([
                transforms.ToTensor()
            ])

        def __len__(self):
            return len(self.submission_df)

        def __getitem__(self,item):
            image_id = self.submission_df['StudyInstanceUID'].values[item]

            image_path = self.submission_df['path'].values[item]
            image = dicom2array(image_path)
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            image = cv2.resize(image, Config.image_size)

            image = albumentations.Compose([albumentations.Normalize()])(image=image)['image']

            image = torch.tensor(image)

            return {
                    "x":image
                    }



    test_dataset = SIIMCOVID19TestData()

    test_loader = DataLoader(test_dataset,
                            batch_size=32,
                            num_workers=8)

# Inference - Efnb7

In [ ]:
if not fast_sub:
    import numpy as np 
    import pandas as pd

    if fast_sub:
        df = fast_df.copy()
    else:
        df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')

    id_laststr_list  = []
    for i in range(df.shape[0]):
        id_laststr_list.append(df.loc[i,'id'][-1])
    df['id_last_str'] = id_laststr_list

    study_len = df[df['id_last_str'] == 'y'].shape[0]

In [ ]:
if not fast_sub:
    strategy = auto_select_accelerator()
    BATCH_SIZE = strategy.num_replicas_in_sync * 16

    IMSIZE = (224, 240, 260, 300, 380, 456, 528, 600, 512)

    if fast_sub:
        sub_df = fast_df.copy()
    else:
        sub_df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')

    sub_df = sub_df[:study_len]
    test_paths = f'/kaggle/tmp/{split}/study/' + sub_df['id'] +'.png'

    sub_df['negative'] = 0
    sub_df['typical'] = 0
    sub_df['indeterminate'] = 0
    sub_df['atypical'] = 0

In [ ]:
if not fast_sub:
    label_cols = sub_df.columns[2:]

    test_decoder = build_decoder(with_labels=False, target_size=(IMSIZE[7], IMSIZE[7]), ext='png')
    dtest = build_dataset(
        test_paths, bsize=BATCH_SIZE, repeat=False, 
        shuffle=False, augment=False, cache=False,
        decode_fn=test_decoder
    )


In [ ]:
if not fast_sub:
    with strategy.scope():

        models = []

        models0 = tf.keras.models.load_model(
            '../input/siim-covid19-efnb7-train-study/model0.h5'
        )
        models1 = tf.keras.models.load_model(
            '../input/siim-covid19-efnb7-train-study/model1.h5'
        )
        models2 = tf.keras.models.load_model(
            '../input/siim-covid19-efnb7-train-study/model2.h5'
        )
        models3 = tf.keras.models.load_model(
            '../input/siim-covid19-efnb7-train-study/model3.h5'
        )
        models4 = tf.keras.models.load_model(
            '../input/siim-covid19-efnb7-train-study/model4.h5'
        )

        models.append(models0)
        models.append(models1)
        models.append(models2)
        models.append(models3)
        models.append(models4)

In [ ]:
if not fast_sub:
    with torch.no_grad():
        preds_efnb7_1 = models0.predict(dtest, verbose=1)
        preds_efnb7_2 = models1.predict(dtest, verbose=1)
        preds_efnb7_3 = models2.predict(dtest, verbose=1)
        preds_efnb7_4 = models3.predict(dtest, verbose=1)
        preds_efnb7_5 = models4.predict(dtest, verbose=1)

In [ ]:
if not fast_sub:
    del models
    del models0, models1, models2, models3, models4
    del dtest, test_decoder
    _ = gc.collect()
    torch.cuda.empty_cache()

In [ ]:
### END HERE ###

# Study level predictions

In [ ]:
# MOST RECENT COMMENT

# # Weighting: BEST CV
# wt_swin = 0.28475899
# wt_efnb3ns = 0.12637422
# wt_efnb7 = 0.21074902
# wt_efnv2l = 0.37811778

# Weighting: BEST LB, interpolated
wt_swin = 0.12743
wt_efnb3ns = 0.1591
wt_efnb7 = 0.21404
wt_efnv2l = 0.49943

wt_efnb7_1 = 0.1
wt_efnb7_2 = 0.1
wt_efnb7_3 = 0.1
wt_efnb7_4 = 0.1
wt_efnb7_5 = 0.6

print(wt_swin + wt_efnb3ns + wt_efnb7 + wt_efnv2l)
print(wt_efnb7_1 + wt_efnb7_2 + wt_efnb7_3 + wt_efnb7_4 + wt_efnb7_5) 

In [ ]:
if not fast_sub:
    submission_df = pd.read_csv("/kaggle/input/siim-covid19-detection/sample_submission.csv")

    study_submission_df = submission_df[submission_df['id'].str.endswith('_study')]
    image_submission_df = submission_df[submission_df['id'].str.endswith('_image')]
    
    negPredLst = []

    for ind, pred in tqdm(enumerate(preds_efnb7_1), total=len(preds_efnb7_1)):

        finalpred = []
        study_id = study_submission_df['id'].iloc[ind] # Looks like 04c42f33c006_study
        study_id_short = study_id[:-len("_study")] # Looks like 04c42f33c006
        
        # Negative
        finalpred.append((
            (wt_efnb7*(wt_efnb7_1*preds_efnb7_1[ind][0] + wt_efnb7_2*preds_efnb7_2[ind][0] + wt_efnb7_3*preds_efnb7_3[ind][0] + wt_efnb7_4*preds_efnb7_4[ind][0] + wt_efnb7_5*preds_efnb7_5[ind][0])) 
            + (wt_efnv2l*effv2l_preds['negative'][study_id]) 
            + (wt_swin*swin_preds['neg'][study_id_short])
            + (wt_efnb3ns*eff3_preds['neg'][study_id_short])
        ))
        
        # Typical
        finalpred.append((
            (wt_efnb7*(wt_efnb7_1*preds_efnb7_1[ind][1] + wt_efnb7_2*preds_efnb7_2[ind][1] + wt_efnb7_3*preds_efnb7_3[ind][1] + wt_efnb7_4*preds_efnb7_4[ind][1] + wt_efnb7_5*preds_efnb7_5[ind][1])) 
            + (wt_efnv2l*effv2l_preds['typical'][study_id]) 
            + (wt_swin*swin_preds['typ'][study_id_short]) 
            + (wt_efnb3ns*eff3_preds['typ'][study_id_short])
        ))

        # Indeterminate
        finalpred.append((
            (wt_efnb7*(wt_efnb7_1*preds_efnb7_1[ind][2] + wt_efnb7_2*preds_efnb7_2[ind][2] + wt_efnb7_3*preds_efnb7_3[ind][2] + wt_efnb7_4*preds_efnb7_4[ind][2] + wt_efnb7_5*preds_efnb7_5[ind][2])) 
            + (wt_efnv2l*effv2l_preds['indeterminate'][study_id]) 
            + (wt_swin*swin_preds['ind'][study_id_short]) 
            + (wt_efnb3ns*eff3_preds['ind'][study_id_short])
        ))

        # Atypical
        finalpred.append((
            (wt_efnb7*(wt_efnb7_1*preds_efnb7_1[ind][3] + wt_efnb7_2*preds_efnb7_2[ind][3] + wt_efnb7_3*preds_efnb7_3[ind][3] + wt_efnb7_4*preds_efnb7_4[ind][3] + wt_efnb7_5*preds_efnb7_5[ind][3])) 
            + (wt_efnv2l*effv2l_preds['atypical'][study_id]) 
            + (wt_swin*swin_preds['atyp'][study_id_short]) 
            + (wt_efnb3ns*eff3_preds['atyp'][study_id_short])
        ))

        negative, typical, indeterminate, atypical = str(finalpred[0]),str(finalpred[1]),str(finalpred[2]),str(finalpred[3])
    #     negative, typical, indeterminate, atypical = str(preds_effnetb0[ind][0]),str(preds_effnetb0[ind][1]),str(preds_effnetb0[ind][2]),str(preds_effnetb0[ind][3])
        study_submission_df.loc[ind,'PredictionString'] = f'negative {negative} 0 0 1 1 typical {typical} 0 0 1 1 indeterminate {indeterminate} 0 0 1 1 atypical {atypical} 0 0 1 1'
        
        negPredLst.append(negative)
    
    print(study_submission_df.shape)


#wt_efnb3ns*preds_efnb3ns[ind][0] + \

In [ ]:
if not fast_sub:
    negPredStudy_df = study_submission_df.copy()
    negPredStudy_df['negPreds'] = negPredLst
    negPredStudy_df = negPredStudy_df[['id','negPreds']]

In [ ]:
if not fast_sub:
    print(negPredStudy_df.shape)
    negPredStudy_df.head()

# 2 class prediction (Opacity/None)

In [ ]:
if not fast_sub:
    if fast_sub:
        sub_df = fast_df.copy()
    else:
        sub_df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')
    sub_df = sub_df[study_len:]
    test_paths = f'/kaggle/tmp/{split}/image/' + sub_df['id'] +'.png'
    sub_df['none'] = 0

    label_cols = sub_df.columns[2]

    test_decoder = build_decoder(with_labels=False, target_size=(IMSIZE[8], IMSIZE[8]), ext='png')
    dtest = build_dataset(
        test_paths, bsize=BATCH_SIZE, repeat=False, 
        shuffle=False, augment=False, cache=False,
        decode_fn=test_decoder
    )

    with strategy.scope():

        models = []
        models0 = tf.keras.models.load_model(
            '/kaggle/input/siim-covid19-efnb7-train-fold0-5-2class/model0.h5'
        )
        models1 = tf.keras.models.load_model(
            '/kaggle/input/siim-covid19-efnb7-train-fold0-5-2class/model1.h5'
        )
        models2 = tf.keras.models.load_model(
            '/kaggle/input/siim-covid19-efnb7-train-fold0-5-2class/model2.h5'
        )
        models3 = tf.keras.models.load_model(
            '/kaggle/input/siim-covid19-efnb7-train-fold0-5-2class/model3.h5'
        )
        models4 = tf.keras.models.load_model(
            '/kaggle/input/siim-covid19-efnb7-train-fold0-5-2class/model4.h5'
        )
        models5 = tf.keras.models.load_model(
            '../input/b77trainyukita/model0.h5'
        )
        models6 = tf.keras.models.load_model(
            '../input/b77trainyukita/model1.h5'
        )
        models7 = tf.keras.models.load_model(
            '../input/b77trainyukita/model2.h5'
        )
        models8 = tf.keras.models.load_model(
            '../input/b77trainyukita/model3.h5'
        )
        models9 = tf.keras.models.load_model(
            '../input/b77trainyukita/model4.h5'
        )
        models10 = tf.keras.models.load_model(
            '../input/restnet152modelsyukit/model3.h5'
        )
        models11 = tf.keras.models.load_model(
            '../input/restnet152modelsyukit/model2.h5'
        )
        models12 = tf.keras.models.load_model(
            '../input/restnet152modelsyukit/model1.h5'
        )
        models13 = tf.keras.models.load_model(
            '../input/restnet152modelsyukit/model4.h5'
        )
        models14 = tf.keras.models.load_model(
            '../input/restnet152modelsyukit/model0.h5'
        )
    #     models15 = tf.keras.models.load_model(
    #         '../input/siimefnmod/model4.h5'
    #     )

        models.append(models0)
        models.append(models1)
        models.append(models2)
        models.append(models3)
        models.append(models4)
        models.append(models5)
        models.append(models6)
        models.append(models7)
        models.append(models8)
        models.append(models9)
        models.append(models10)
        models.append(models11)
        models.append(models12)
        models.append(models13)
        models.append(models14)
    #     models.append(models15)

    weights = {
        0: 3.125,
        1: 2,
        2: 2,
        3: 2,
        4: 2,
        5: 2,
        6: 1,
        7: 1,
        8: 1,
        9: 2,
        10: 0,
        11: 1.125,
        12: 0,
        13: 0,
        14: 1.125

    }

    weights_sum = sum(weights.values())
    weights = {k: v/weights_sum for k, v in weights.items()}

    predictions = [model.predict(dtest, verbose=1) for model in models]
    for i, pred in enumerate(predictions):
        predictions[i] = weights[i] * pred

    sub_df[label_cols] = sum(predictions)
    #sub_df[label_cols] = sum([model.predict(dtest, verbose=1) for model in models]) / len(models)
    df_2class = sub_df.reset_index(drop=True)

In [ ]:
if not fast_sub:
    import glob

    files = []

    for file in glob.glob("../input/siim-covid19-detection/test/*/*/*"):
        files.append(file)

In [ ]:
if not fast_sub:
    test_study_image_df = pd.DataFrame()

    test_study = []
    test_image = []

    for ind, file in tqdm(enumerate(files)):
        test_study.append(files[ind].split('/')[4] + "_study")
        test_image.append(files[ind].split('/')[-1].split('.')[0] + "_image")


    test_study_image_df['study_id'] = test_study
    test_study_image_df['image_id'] = test_image

    print(test_study_image_df.shape)
    test_study_image_df.head()

In [ ]:
if not fast_sub:
    temp1 = test_study_image_df.rename(columns={'image_id':'id'})
    df_2class = pd.merge(df_2class, temp1, on="id", how="left")

    df_2class.head()

In [ ]:
if not fast_sub:
    temp2 = negPredStudy_df.rename(columns={'id':'study_id'})
    df_2class = pd.merge(df_2class, temp2, on="study_id", how="left")

    print(df_2class.shape)
    df_2class.head()

In [ ]:
if not fast_sub:
    df_2class['negPreds'] = df_2class['negPreds'].astype(float)
#     df_2class['none'] = df_2class[['none', 'negPreds']].mean(axis=1)
    df_2class['none'] = (0.5*df_2class['none']) + (0.5*df_2class['negPreds'])

    df_2class.drop(columns = ['study_id','negPreds'], inplace=True)

    df_2class.head()

In [ ]:
if not fast_sub:
    del models, negPredStudy_df, temp1, temp2, negPredLst
    del models0, models1, models2, models3, models4, models5, models6, models7, models8, models9, models10, models11, models12, models13, models14
    del dtest, test_decoder
    _ = gc.collect()

In [ ]:
if not fast_sub:
    from numba import cuda
    import torch
    cuda.select_device(0)
    cuda.close()
    cuda.select_device(0)

# YOLOv5: Image level Object detection

In [ ]:
if not fast_sub:
    import numpy as np, pandas as pd
    from glob import glob
    import shutil, os
    import matplotlib.pyplot as plt
    from sklearn.model_selection import GroupKFold
    from tqdm.notebook import tqdm
    import seaborn as sns
    import torch

In [ ]:
if not fast_sub:
    meta = meta[meta['split'] == 'test']
    if fast_sub:
        test_df = fast_df.copy()
    else:
        test_df = pd.read_csv('/kaggle/input/siim-covid19-detection/sample_submission.csv')
    test_df = df[study_len:].reset_index(drop=True) 
    meta['image_id'] = meta['image_id'] + '_image'
    meta.columns = ['id', 'dim0', 'dim1', 'split']
    test_df = pd.merge(test_df, meta, on = 'id', how = 'left')

In [ ]:
if not fast_sub:
    dim = 512 #1024, 256, 'original'
    test_dir = f'/kaggle/tmp/{split}/image'
    #weights_dir = '/kaggle/input/siim-cov19-yolov5-train/yolov5/runs/train/exp/weights/best.pt'
    weights_dir = '/kaggle/input/weights-of-yolov5-150-epochs/best.pt'

    shutil.copytree('/kaggle/input/yolov5-repo/yolov5-master', '/kaggle/working/yolov5')
    os.chdir('/kaggle/working/yolov5') # install dependencies
    !git pull

    import torch
    #from IPython.display import Image, clear_output  # to display images

    #clear_output()
    #print('Setup complete. Using torch %s %s' % (torch.__version__, torch.cuda.get_device_properties(0) if torch.cuda.is_available() else 'CPU'))


    !python detect.py --weights {weights_dir}\
    --img 512\
    --conf 0.0001\
    --iou 0.5\
    --augment\
    --source {test_dir}\
    --save-txt --save-conf --exist-ok

    # --max-det 40\

    def yolo2voc(image_height, image_width, bboxes):
        """
        yolo => [xmid, ymid, w, h] (normalized)
        voc  => [x1, y1, x2, y1]

        """ 
        bboxes = bboxes.copy().astype(float) # otherwise all value will be 0 as voc_pascal dtype is np.int

        bboxes[..., [0, 2]] = bboxes[..., [0, 2]]* image_width
        bboxes[..., [1, 3]] = bboxes[..., [1, 3]]* image_height

        bboxes[..., [0, 1]] = bboxes[..., [0, 1]] - bboxes[..., [2, 3]]/2
        bboxes[..., [2, 3]] = bboxes[..., [0, 1]] + bboxes[..., [2, 3]]

        return bboxes
    image_ids = []
    PredictionStrings = []

    for file_path in tqdm(glob('runs/detect/exp/labels/*.txt')):
        image_id = file_path.split('/')[-1].split('.')[0]
        w, h = test_df.loc[test_df.id==image_id,['dim1', 'dim0']].values[0]
        f = open(file_path, 'r')
        data = np.array(f.read().replace('\n', ' ').strip().split(' ')).astype(np.float32).reshape(-1, 6)
        data = data[:, [0, 5, 1, 2, 3, 4]]
        bboxes = list(np.round(np.concatenate((data[:, :2], np.round(yolo2voc(h, w, data[:, 2:]))), axis =1).reshape(-1), 12).astype(str))
        for idx in range(len(bboxes)):
            bboxes[idx] = str(int(float(bboxes[idx]))) if idx%6!=1 else bboxes[idx]
        image_ids.append(image_id)
        PredictionStrings.append(' '.join(bboxes))


    pred_df = pd.DataFrame({'id':image_ids,
                            'PredictionString':PredictionStrings})

In [ ]:
if not fast_sub:
    test_df = test_df.drop(['PredictionString'], axis=1)
    sub_df = pd.merge(test_df, pred_df, on = 'id', how = 'left').fillna("none 1 0 0 1 1")
    sub_df = sub_df[['id', 'PredictionString']]
    for i in range(sub_df.shape[0]):
        if sub_df.loc[i,'PredictionString'] == "none 1 0 0 1 1":
            continue
        sub_df_split = sub_df.loc[i,'PredictionString'].split()
        sub_df_list = []
        for j in range(int(len(sub_df_split) / 6)):
            sub_df_list.append('opacity')
            sub_df_list.append(sub_df_split[6 * j + 1])
            sub_df_list.append(sub_df_split[6 * j + 2])
            sub_df_list.append(sub_df_split[6 * j + 3])
            sub_df_list.append(sub_df_split[6 * j + 4])
            sub_df_list.append(sub_df_split[6 * j + 5])
        sub_df.loc[i,'PredictionString'] = ' '.join(sub_df_list)
    sub_df['none'] = df_2class['none'] 
    for i in range(sub_df.shape[0]):
        if sub_df.loc[i,'PredictionString'] != 'none 1 0 0 1 1':
            sub_df.loc[i,'PredictionString'] = sub_df.loc[i,'PredictionString'] + ' none ' + str(sub_df.loc[i,'none']) + ' 0 0 1 1'
    sub_df = sub_df[['id', 'PredictionString']]   

In [ ]:
if not fast_sub:
    sub_df.shape

# Final submission file

In [ ]:
if not fast_sub:
    !rm -r /kaggle/working/mmdetection
    shutil.rmtree('/kaggle/working/yolov5')
else:
    !rm -r /kaggle/working/mmdetection

In [ ]:
if not fast_sub:
    submission_df = study_submission_df.append(sub_df).reset_index(drop=True)
    submission_df.to_csv('/kaggle/working/submission.csv',index = False)  

In [ ]:
if not fast_sub:
    print(submission_df.shape)

In [ ]:
if not fast_sub:
    display(submission_df.head())

In [ ]:
if not fast_sub:
    display(submission_df.tail())